<a href="https://colab.research.google.com/github/devarahaasan/Aerial-Object-Classification-Detection/blob/main/project5e.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Streamlit

In [1]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 66.8 MB/s eta 0:00:00
--2026-03-30 10:09:08--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-03-30 10:09:08--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-30T10%3A50%3A37Z&rscd=attachment%3B+filename%3Dclou

In [2]:
!pip install streamlit-option-menu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 12.7 MB/s eta 0:00:00


In [3]:
pip install tf-keras

In [4]:
!pip install tf-keras streamlit-option-menu

In [5]:
!pip install tf-keras streamlit pillow numpy

In [19]:
!pkill streamlit

In [20]:
%%writefile app8.py
import streamlit as st
import tensorflow as tf
from PIL import Image
import numpy as np
import os

# 1. (Modern Dark Theme)
st.set_page_config(page_title="Aerial Object Classifier", page_icon="🦅", layout="centered")

st.markdown("""
    <style>
    .stApp {
        background-color: #0E1117;
        color: #FFFFFF;
    }
    .stButton>button {
        background: linear-gradient(45deg, #00f2fe 0%, #4facfe 100%);
        color: white;
        border: none;
        padding: 15px 32px;
        border-radius: 12px;
        font-weight: bold;
        width: 100%;
        font-size: 18px;
        transition: 0.3s ease;
    }
    .stButton>button:hover {
        transform: translateY(-2px);
        box-shadow: 0px 8px 20px rgba(79, 172, 254, 0.4);
    }
    .result-box {
        background-color: #161B22;
        padding: 30px;
        border-radius: 20px;
        border: 1px solid #30363D;
        text-align: center;
        margin-top: 20px;
    }
    </style>
    """, unsafe_allow_html=True)

# 2. load model
@st.cache_resource
def load_custom_model():
    model_path = 'aerial_finetuned_final.keras'
    if not os.path.exists(model_path):
        st.error(f"❌ '{model_path}' if not found ,upload the file.")
        return None

    return tf.keras.models.load_model(model_path)

model = load_custom_model()

# 3. UI tittle
st.title("🦅 Aerial Object Classification")
st.markdown("<p style='color: #8B949E;'>Custom CNN Model: Detecting Birds vs Drones</p>", unsafe_allow_html=True)
st.divider()

# 4. upload image
file = st.file_uploader("choose images(JPG/PNG)", type=["jpg", "png", "jpeg"])

if file is not None:
    image = Image.open(file).convert('RGB')
    st.image(image, caption="📸 Uploaded Image", use_container_width=True)

    #  (224x224 and Normalization)
    img = image.resize((224, 224))
    img_array = np.array(img).astype('float32') / 255.0
    img_reshape = np.expand_dims(img_array, axis=0)

    if st.button('🚀 START AI DETECTION'):
        if model is not None:
            with st.spinner('AI searching...'):
                prediction = model.predict(img_reshape)
                score = float(prediction[0][0])

                st.markdown("<div class='result-box'>", unsafe_allow_html=True)

                # Binary Logic: 0.5 Threshold
                # (Bird = 0, Drone = 1 )
                if score > 0.5:
                    st.markdown("<h2 style='color: #00f2fe;'>Result: DRONE</h2>", unsafe_allow_html=True)
                    st.write(f"Confidence Level: **{score*100:.2f}%**")
                    st.progress(score)
                else:
                    st.markdown("<h2 style='color: #FC00FF;'>Result: BIRD</h2>", unsafe_allow_html=True)
                    st.write(f"Confidence Level: **{(1-score)*100:.2f}%**")
                    st.progress(1 - score)

                st.markdown("</div>", unsafe_allow_html=True)
        else:
            st.error("if not load!")

st.markdown("<br><hr><center><p style='color: #555;'>Project by Devahaasan | Custom CNN Build</p></center>", unsafe_allow_html=True)

Overwriting app8.py


In [21]:
!streamlit run /content/app8.py &>/content/logs.txt &

In [22]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://fundamental-installation-dual-seminars.trycloudflare.com
